# 03 · Synchronized player relationships

**Feature implementation and analytic checks — not an NFL accuracy result.**

The proposed signal is how relative movement changes *during the observed window*,
not a nearest-defender label. Each edge compares two players at the same frame.
An absent observation breaks the edge; a stale observation is never moved to the throw.

This representation is opt-in and is **not connected to the frozen coordinate/velocity
experiment**. The feature-research gate remains open. The existing scientific run must
be reviewed before fitting another treatment. See [the protocol](../docs/RELATION_HISTORY.md).

The example below is two invented linear trajectories. It tests geometry, masks and
relative time, not whether a model can forecast real NFL players.

In [1]:
import json
import sys

import numpy as np
import plotly.graph_objects as go
from IPython.display import display

from nfl_trajectory.features import ROLES
from nfl_trajectory.relation_history import RELATION_NAMES, pair_history
from nfl_trajectory.temporal_data import CHANNELS, HISTORY

clock = np.arange(1 - HISTORY, 1, dtype=np.float32) / 10
history = np.zeros((2, HISTORY, len(CHANNELS)), dtype=np.float32)
channels = list(CHANNELS)
for player, (x0, y0, vx) in enumerate(((40.0, 20.0, 2.0), (43.0, 24.0, 1.0))):
    history[player, :, channels.index("ball_dx")] = (55 - x0 - vx * clock) / 20
    history[player, :, channels.index("ball_dy")] = (25 - y0) / 20
    history[player, :, channels.index("vx")] = vx / 10
observed = np.ones((2, HISTORY), dtype=bool)
observed[1, 6:9] = False
observed[1, -3:] = False
side = np.array([1, 0], dtype=np.int64)
role = np.array([ROLES.index("Targeted Receiver"), ROLES.index("Defensive Coverage")])
relations = pair_history(history, observed, side, role)
terminal = relations.terminal_view()
assert relations.observed.sum() == 28
assert terminal.observed.sum() == 2
assert terminal.observed[0, 1, -4]
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | synthetic inputs only")
print("Feature tensor:", relations.values.shape, "| channels:", len(RELATION_NAMES))
print("Supported directed edge-frames:", int(relations.observed.sum()))
print("Last shared observation: -0.3 seconds; NOT retimed to the throw.")

Python 3.13.5 | NumPy 2.3.5 | synthetic inputs only
Feature tensor: (2, 2, 20, 12) | channels: 12
Supported directed edge-frames: 28
Last shared observation: -0.3 seconds; NOT retimed to the throw.


## Observed separation with genuine gaps

Distance uses a shared coordinate origin: subtract the two ball-relative offsets.
Subtracting two players' individually centered displacement histories would be wrong.
The ball landing point is organizer-supplied and cancels in this calculation.

The line intentionally breaks where either track is absent. No interpolation, forward
fill, forecast target, or retrospective coverage annotation is used.

In [2]:
support = relations.observed[0, 1]
separation = np.where(support, relations.values[0, 1, :, 4] * 20, np.nan)
expected = np.sqrt((3 - clock) ** 2 + 4**2)
np.testing.assert_allclose(separation[support], expected[support], atol=2e-6)
fig = go.Figure(
    go.Scatter(
        x=clock.tolist(),
        y=separation.tolist(),
        mode="lines+markers",
        connectgaps=False,
        name="Observed pair separation",
    )
)
fig.update_layout(
    template=None,
    title="Synthetic receiver–defender pair: synchronized observations only",
    xaxis_title="Seconds relative to observed cutoff",
    yaxis_title="Separation (yards)",
    height=360,
)
display({"application/vnd.plotly.v1+json": json.loads(fig.to_json())}, raw=True)
print("seconds   separation_yards   supported")
for second, distance, present in zip(clock, separation, support, strict=True):
    print(f"{second:7.1f}   {distance:16.4f}   {bool(present)}")

seconds   separation_yards   supported
   -1.9             6.3253   True
   -1.8             6.2482   True
   -1.7             6.1717   True
   -1.6             6.0959   True
   -1.5             6.0208   True
   -1.4             5.9464   True
   -1.3                nan   False
   -1.2                nan   False
   -1.1                nan   False
   -1.0             5.6569   True
   -0.9             5.5866   True
   -0.8             5.5172   True
   -0.7             5.4489   True
   -0.6             5.3815   True
   -0.5             5.3151   True
   -0.4             5.2498   True
   -0.3             5.1856   True
   -0.2                nan   False
   -0.1                nan   False
    0.0                nan   False


## Leakage and availability check

Unobserved entries may contain arbitrary values, even NaN, without changing any edge.
Only four observed channel families plus organizer roles and sides enter the function.
There is no argument for future player positions. Self-edges are masked explicitly.

A terminal-only view retains the exact same tensor shape and the actual time of the
last *joint* observation. This supports a future capacity-matched ablation; it does not
yet include a trained temporal relationship encoder or a measured RMSE improvement.

In [3]:
poisoned = history.copy()
poisoned[~observed] = np.nan
rechecked = pair_history(poisoned, observed, side, role)
np.testing.assert_array_equal(rechecked.values, relations.values)
assert not relations.observed[np.arange(2), np.arange(2)].any()
assert np.isfinite(relations.values).all()
print("PASS: masked nonfinite observations do not affect features.")
print("PASS: analytic distance and joint-observation timing agree.")
print("Scientific fits: 0 | NFL validation RMSE: not measured | Feature gate: OPEN")

PASS: masked nonfinite observations do not affect features.
PASS: analytic distance and joint-observation timing agree.
Scientific fits: 0 | NFL validation RMSE: not measured | Feature gate: OPEN
